# HMP-GNN — Google Colab

**仅主实验**（`main.py` 中 `main()` 的 `config`；可选 `COLAB_CONFIG_OVERRIDES`）。

1. GPU：Runtime → Change runtime type → T4 等。  
2. 先 **Step 0** 取代码。  
3. **Step 3** 跑 FL。  
4. **Step 4** 内联 **Figure 1–5**、**CSE（Semantic entropy）演化图**、**Perplexity JSON 摘要**，以及可选下游产物提示。  
5. **Step 5** 打印完整 **config / progressive_metrics / 每轮日志 / PPL / M7** 等，便于核对与复制。  
6. **Step 6** 可选 zip 本机下载；**Step 6b** 可选将 **zip +（可选）完整 `results/`** 复制到 **Google Drive**。  
7. **Step 7** 释放 **GPU**。


## Step 0: 获取代码

若当前目录下已有 `main.py`，不克隆；否则拉取并进入 `HMP-GNN`。

In [ ]:
# Fetch repository and set working directory
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/GuangLun2000/HMP-GNN.git"
REPO_DIR = Path("HMP-GNN")

def code_files_present():
    return Path("main.py").is_file() and Path("data_loader.py").is_file()

if code_files_present():
    print("已在仓库根目录（发现 main.py）。")
else:
    if REPO_DIR.is_dir():
        print(f"使用已有目录: {REPO_DIR}")
    else:
        print(f"正在克隆 {REPO_URL} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir(REPO_DIR)
    print(f"工作目录: {Path('.').resolve()}")

sys.path.insert(0, str(Path('.').resolve()))
print("cwd:", Path('.').resolve())


## Step 1: 安装依赖

In [ ]:
from pathlib import Path
import subprocess
import sys

req = Path("requirements.txt")
if req.is_file():
    print("Installing from requirements.txt ...")
    %pip install -q -r requirements.txt
else:
    %pip install -q torch transformers datasets numpy scikit-learn pandas tqdm matplotlib seaborn peft "torchao>=0.17"
# Colab 常预装 torchao 0.10；PEFT 注入 LoRA 时若版本 <0.16 会直接 ImportError
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "torchao>=0.17"],
    check=False,
)
print("依赖已安装")


## Step 2: 检查文件与 GPU

In [ ]:
import os
from pathlib import Path
import torch

required = [
    "main.py", "client.py", "server.py", "data_loader.py", "models.py", "visualization.py",
    "defense/__init__.py", "fed_checkpoint.py", "evaluation_hallucination.py",
    "attack/hallucination.py", "attack/__init__.py", "hmp_gae/__init__.py",
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print("缺少文件:", missing)
else:
    print("核心文件已就绪")
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Step 3: 运行主实验

**参数**仅来自 `main.py` 的 `config`，此处仅有可选的 **`COLAB_CONFIG_OVERRIDES`**。

In [ ]:
# 可选：合并进 main() 的 config；空 {} = 使用 main.py 原样
COLAB_CONFIG_OVERRIDES = {
    # "experiment_name": "my_colab_run",
    # "num_rounds": 5,
    # "dataset_size_limit": 8000,
}

print("Configuration: main.py 内 config")
if COLAB_CONFIG_OVERRIDES:
    print("  + Colab 覆写键:", list(COLAB_CONFIG_OVERRIDES.keys()))
else:
    print("  + Colab 覆写: 无")


In [ ]:
import os
import gc
import warnings
import torch
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from main import main

print("启动实验（main.py 默认 config，除非 COLAB_CONFIG_OVERRIDES 非空）")
print("=" * 60)
try:
    main(config_overrides=COLAB_CONFIG_OVERRIDES if COLAB_CONFIG_OVERRIDES else None)
    print()
    print("实验完成")
except Exception as e:
    print()
    print("失败:", e)
    import traceback
    traceback.print_exc()
finally:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("gc + CUDA empty_cache 已执行")


## Step 4: 主实验图 + CSE + PPL

- **Figure 1–5**：`{experiment_name}_figure{i}.png`  
- **CSE**：`visualization.plot_cse_evolution` 生成 `{experiment_name}_figure_cse_evolution.png`（失败时以内联 matplotlib 曲线 + 表格代替）  
- **PPL**：展示 `{experiment_name}_eval_ppl.json`（需 config 中 `eval_perplexity` 与 `save_global_checkpoint` 且为 decoder 骨干）

In [ ]:
from IPython.display import Image, display, Markdown, JSON
from pathlib import Path
import json

results_dir = Path("results")
candidates = sorted(
    results_dir.glob("*_results.json"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
if not candidates:
    display(Markdown("未找到 `results/*_results.json`。"))
else:
    rpath = candidates[0]
    with open(rpath, "r", encoding="utf-8") as f:
        bundle = json.load(f)
    cfg = bundle.get("config") or {}
    exp = cfg.get("experiment_name", "experiment")
    display(
        Markdown(
            "**主结果文件:** `"
            + rpath.name
            + "` &nbsp;|&nbsp; **experiment_name:** `"
            + str(exp)
            + "`"
        )
    )
    for i in range(1, 6):
        p = results_dir / f"{exp}_figure{i}.png"
        if p.is_file():
            display(Markdown("#### Figure " + str(i) + " — `" + p.name + "`"))
            display(Image(filename=str(p)))
        else:
            display(Markdown("#### Figure " + str(i) + " — 未找到"))

    # —— Classification Semantic Entropy (CSE) ——
    display(Markdown("### Classification Semantic Entropy (CSE)"))
    pm = bundle.get("progressive_metrics") or {}
    cse_png = results_dir / f"{exp}_figure_cse_evolution.png"
    try:
        from visualization import plot_cse_evolution

        plot_cse_evolution(
            {str(exp): rpath},
            save_path=cse_png,
            title_suffix="Colab",
            x_attack_start=cfg.get("attack_start_round"),
        )
    except Exception as e:
        display(Markdown(f"*调用 `plot_cse_evolution` 失败:* `{type(e).__name__}: {e}` *，将尝试内联曲线。*"))

    if cse_png.is_file():
        display(Markdown("#### CSE 演化图 — `" + cse_png.name + "`"))
        display(Image(filename=str(cse_png)))
    else:
        rs = pm.get("rounds") or []
        cs = pm.get("cse") or []
        pairs = []
        for r, c in zip(rs, cs):
            if c is None or isinstance(c, bool):
                continue
            if isinstance(c, (int, float)) and c == c:  # not NaN
                pairs.append((int(r), float(c)))
        if pairs:
            import matplotlib.pyplot as plt

            fig, ax = plt.subplots(figsize=(6.5, 4))
            xs, ys = zip(*pairs)
            ax.plot(xs, ys, marker="o", linewidth=1.6, color="#0B6E4F")
            ax.set_xlabel("Communication Round")
            ax.set_ylabel(r"CSE $H(p(y|x))$ (nats)")
            ax.set_title("Classification Semantic Entropy")
            ax.grid(True, alpha=0.3)
            fig.tight_layout()
            display(fig)
            plt.close(fig)
        else:
            display(
                Markdown(
                    "*无可用 CSE 曲线数据。* 请确认 `eval_classification_semantic_entropy` 为 True，"
                    "且 `progressive_metrics.cse` 含数值。"
                )
            )

    rs = pm.get("rounds") or []
    cs = pm.get("cse") or []
    pairs_tbl = [
        (int(r), float(c))
        for r, c in zip(rs, cs)
        if c is not None and isinstance(c, (int, float)) and not isinstance(c, bool) and c == c
    ]
    if pairs_tbl:
        display(Markdown("#### 每轮 CSE（表格）"))
        md = "| Round | CSE (nats) |\n|---:|---:|\n"
        for r, c in pairs_tbl:
            md += f"| {r} | {c:.6f} |\n"
        display(Markdown(md))

    # —— Perplexity (eval_ppl) ——
    display(Markdown("### Perplexity（`eval_perplexity`）"))
    ppl_path = results_dir / f"{exp}_eval_ppl.json"
    if ppl_path.is_file():
        with open(ppl_path, "r", encoding="utf-8") as f:
            ppl_data = json.load(f)
        display(Markdown("#### `" + ppl_path.name + "`"))
        display(JSON(ppl_data))
        if not ppl_data.get("skipped"):
            pmv = ppl_data.get("ppl_mean")
            ns = ppl_data.get("n_samples")
            display(Markdown(f"**PPL mean:** `{pmv}` &nbsp;|&nbsp; **n_samples:** `{ns}`"))
        else:
            display(Markdown(f"*PPL 已跳过:* `{ppl_data.get('skip_reason')}`"))
    else:
        display(Markdown(f"*未找到* `{ppl_path.name}`。"))
        display(
            Markdown(
                "若需要 PPL：在 `main.py` 或 `COLAB_CONFIG_OVERRIDES` 中保持 "
                "`eval_perplexity=True`、`save_global_checkpoint=True`，并使用 **decoder** 骨干（如 Qwen2）。"
            )
        )
        display(
            Markdown(
                f"当前结果内 config：`eval_perplexity={cfg.get('eval_perplexity')}`，"
                f"`save_global_checkpoint={cfg.get('save_global_checkpoint')}`。"
            )
        )

    # 可选：下游生成产物（仅提示路径 + 前几行）
    ds = results_dir / f"{exp}_downstream_gen.jsonl"
    if ds.is_file():
        display(Markdown("### 下游生成 JSONL（Task 2）"))
        display(Markdown(f"路径: `{ds}` — 完整内容请用 Step 6 zip 下载。"))
        try:
            head = "\n".join(ds.read_text(encoding="utf-8", errors="replace").splitlines()[:8])
            display(Markdown("```jsonl\n" + head + "\n...\n```"))
        except Exception as ex:
            print("预览失败:", ex)



## Step 5: 主实验 — 完整结果（全部在下列单元格中打印）

将展示：合并后的 **config**、**progressive_metrics（含 JSON + 轮次对齐摘要表）**、**每轮** server 指标与 aggregation、**local_accuracies**、**PPL JSON**、**M7 汇总**、**results/** 中相关文件列表（含 `.pt` 检查点，若存在）。


In [ ]:
from __future__ import annotations
import json
from pathlib import Path
from IPython.display import display, Markdown, JSON, Image

R = Path("results")
res_files = sorted(R.glob("*_results.json"), key=lambda p: p.stat().st_mtime, reverse=True)
if not res_files:
    print("未找到 results/*_results.json，请先跑 Step 3。")
else:
    rpath = res_files[0]
    print("=" * 72)
    print("主结果文件:", rpath.resolve())
    with open(rpath, "r", encoding="utf-8") as f:
        data = json.load(f)
    cfg = data.get("config") or {}
    exp = cfg.get("experiment_name", "experiment")
    display(Markdown("### ① 完整 config（与 main 合并后，可展开）"))
    display(JSON(cfg))

    display(Markdown("### ② progressive_metrics"))
    display(JSON(data.get("progressive_metrics") or {}))

    pm = data.get("progressive_metrics") or {}
    display(Markdown("### ②b 轮次对齐摘要（accuracy / CSE / agg_norm）"))
    rnds = pm.get("rounds") or []
    accs = pm.get("clean_acc") or []
    cses = pm.get("cse") or []
    aggs = pm.get("agg_update_norm") or []
    n = max(len(rnds), len(accs), len(cses), len(aggs))
    if n == 0:
        display(Markdown("*progressive_metrics 为空。*"))
    else:
        md = "| idx | round | clean_acc | cse | agg_update_norm |\n|---:|---:|---:|---:|---:|\n"
        for i in range(n):
            r = rnds[i] if i < len(rnds) else ""
            a = accs[i] if i < len(accs) else ""
            c = cses[i] if i < len(cses) else ""
            g = aggs[i] if i < len(aggs) else ""
            if isinstance(a, (int, float)):
                a = f"{a:.6f}"
            if isinstance(c, (int, float)):
                c = f"{c:.6f}"
            elif c is None:
                c = ""
            if isinstance(g, (int, float)):
                g = f"{g:.6f}"
            md += f"| {i + 1} | {r} | {a} | {c} | {g} |\n"
        display(Markdown(md))

    rounds = data.get("results") or []
    display(Markdown("### ③ 每轮指标 + aggregation 可序列化子集"))
    for r in rounds:
        ca = r.get("clean_accuracy")
        gl = r.get("global_loss")
        cse = r.get("classification_semantic_entropy")
        an = (r.get("aggregation") or {}).get("aggregated_update_norm", "")
        print(
            f"Round {r.get('round')}  acc={ca}  loss={gl}  cse={cse}  |acc_diff|={r.get('acc_diff')}  agg_norm={an}"
        )
        agg = r.get("aggregation")
        if not isinstance(agg, dict):
            continue
        slim = {k: agg[k] for k in ("defense", "n_clients", "mode", "aggregated_update_norm", "client_ids") if k in agg}
        if slim:
            try:
                t = json.dumps(slim, ensure_ascii=False, default=str, indent=2)
                print(t if len(t) < 4000 else t[:4000] + "…(truncated)")
            except Exception as ex:
                print("aggregation 序列化略过:", ex)

    loc = data.get("local_accuracies") or {}
    display(Markdown("### ④ local_accuracies（每客户每轮）"))
    for cid, series in sorted(loc.items(), key=lambda x: int(x[0]) if str(x[0]).isdigit() else 0):
        ser = series or []
        head = " ".join(f"{a:.4f}" for a in ser[:32])
        tail = " …" if len(ser) > 32 else ""
        print("  client", cid, "n_rounds=%d" % len(ser), "→", head + tail)

    ppl_path = R / f"{exp}_eval_ppl.json"
    display(Markdown("### ⑤ PPL JSON（`eval_perplexity` + checkpoint；encoder-only 时可能 skipped）"))
    if ppl_path.is_file():
        with open(ppl_path, "r", encoding="utf-8") as f:
            ppl_obj = json.load(f)
        display(JSON(ppl_obj))
        print(
            "[PPL 摘要] skipped=%s ppl_mean=%s n_samples=%s reason=%s"
            % (
                ppl_obj.get("skipped"),
                ppl_obj.get("ppl_mean"),
                ppl_obj.get("n_samples"),
                ppl_obj.get("skip_reason"),
            )
        )
    else:
        print("无:", ppl_path)

    try:
        from visualization import summarize_run_multi_metric
        s = summarize_run_multi_metric(
            rpath, ppl_json_path=ppl_path if ppl_path.is_file() else None
        )
        display(Markdown("### ⑥ M7 多指标汇总结论"))
        print(json.dumps(s, indent=2, default=str, ensure_ascii=False))
    except Exception as e:
        print("M7 汇总异常:", e)

    cse_fig = R / f"{exp}_figure_cse_evolution.png"
    display(Markdown("### ⑥b CSE 演化图（若已保存到 results）"))
    if cse_fig.is_file():
        display(Image(filename=str(cse_fig)))
    else:
        display(Markdown("*未找到 `" + cse_fig.name + "`（可在 Step 4 生成或检查 CSE 数据）。*"))

    display(Markdown("### ⑦ results/ 内与本 `experiment_name` 或数据/图相关的文件"))
    for p in sorted(R.rglob("*")):
        if not p.is_file() or p.name.startswith("."):
            continue
        if "hmp_gae_colab_bundle" in p.parts:
            continue
        if exp in p.name or p.suffix in (".png", ".pdf", ".json", ".jsonl", ".csv", ".pt"):
            print(" ", p.relative_to(R))
    print("=" * 72)


## Step 6: 打包下载 results 产物

In [ ]:
import shutil
from pathlib import Path

out_dir = Path("results/hmp_gae_colab_bundle")
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
root = Path("results")
for p in list(root.rglob("*")):
    if p.is_file() and p.suffix in {".png", ".pdf", ".json", ".jsonl", ".csv"}:
        if "hmp_gae_colab_bundle" in p.parts:
            continue
        dest = out_dir / p.relative_to(root)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dest)
zip_path = str(shutil.make_archive(str(out_dir), "zip", root_dir=out_dir))
print("Zip:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("非 Colab 或未自动下载, zip 路径:", zip_path, type(e).__name__)


## Step 7: 释放 GPU 并断开 Colab 运行时

**务必运行下一单元**，避免长时间占用 GPU。

In [ ]:
import gc, time
import torch

print("准备释放显存并断开本 Colab 运行时…")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.synchronize()
    except Exception:
        pass
time.sleep(2)
try:
    from google.colab import runtime
    print("约 20 秒后 runtime.unassign() 释放实例。")
    time.sleep(20)
    runtime.unassign()
except Exception as e:
    print("非 Colab 或 unassign 不可用:", type(e).__name__, e)
    print("可手动: Runtime -> Disconnect and delete runtime")
